In [2]:
import lightgbm

from config import tick_size
from data_loader import CONFIG_FILES_DIR
from src.data_loader import csv_to_parquet, RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR
from src.config import start_date
from features import MarketDemandModel_data_transformation
import config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from pathlib import Path
from category_encoders import TargetEncoder

import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

C:\Users\UserB\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_parquet(PROCESSED_DATA_DIR / 'train.parquet')

In [4]:
X_train , y_train = MarketDemandModel_data_transformation(df)

In [4]:
print(X_train)

            StockCode  Quantity  Revenue  Unique_Customers  Holiday_flag  \
InvoiceDate                                                                
2009-12-01      10002       143   121.55                11             0   
2009-12-01      20685       167  1077.49                38             0   
2009-12-01      22254        39    48.75                 4             0   
2009-12-01      22255        20    17.00                 3             0   
2009-12-01     85099B      1222  2213.10                54             0   
...               ...       ...      ...               ...           ...   
2011-08-30      82567        19    31.80                 2             0   
2011-08-30      21154        12    29.21                 2             0   
2011-08-30      82578        11     9.55                 3             0   
2011-08-30      23033        24    34.80                 3             0   
2011-08-30       POST       815  1488.82                24             0   

           

In [4]:
len(X_train[X_train['ticks_since_last_sale'] == 0]) / len(X_train)

0.84119007509351

In [14]:
def objective(trial):
    tscv = TimeSeriesSplit(n_splits=5)
    param = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': 1000,

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'cat_smooth': trial.suggest_float('cat_smooth', 10.0, 50.0)
    }

    cv_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMRegressor(**param)


        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )

        preds = model.predict(X_val)
        fold_mae = mean_absolute_error(y_val, preds)
        cv_scores.append(fold_mae)

    return np.mean(cv_scores)

print("Starting Optuna optimization...")
study = optuna.create_study(direction='minimize')

study.optimize(objective, n_trials=50)

print("========================================")
print(f"Best Validation MAE: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")
print("========================================")

[I 2026-03-02 14:35:51,407] A new study created in memory with name: no-name-14afeb39-2f34-4182-9471-50f4cb075425


Starting Optuna optimization...


[I 2026-03-02 14:35:57,209] Trial 0 finished with value: 0.9546193731469751 and parameters: {'learning_rate': 0.1350529362658377, 'num_leaves': 101, 'max_depth': 7, 'min_child_samples': 89, 'colsample_bytree': 0.7700293330269984, 'cat_smooth': 30.056124412681072}. Best is trial 0 with value: 0.9546193731469751.
[I 2026-03-02 14:36:02,122] Trial 1 finished with value: 0.9528097962622596 and parameters: {'learning_rate': 0.06199981489994671, 'num_leaves': 101, 'max_depth': 4, 'min_child_samples': 10, 'colsample_bytree': 0.6653465498640655, 'cat_smooth': 31.4583094879513}. Best is trial 1 with value: 0.9528097962622596.
[I 2026-03-02 14:36:04,974] Trial 2 finished with value: 0.9536228598869343 and parameters: {'learning_rate': 0.17516317190991118, 'num_leaves': 56, 'max_depth': 5, 'min_child_samples': 62, 'colsample_bytree': 0.8213525490788447, 'cat_smooth': 37.632973687281606}. Best is trial 1 with value: 0.9528097962622596.
[I 2026-03-02 14:36:07,381] Trial 3 finished with value: 0.968

Best Validation MAE: 0.9425
Best Parameters:
    'learning_rate': 0.027089223062078577,
    'num_leaves': 134,
    'max_depth': 12,
    'min_child_samples': 49,
    'colsample_bytree': 0.622350510030351,
    'cat_smooth': 35.99779170140813,


In [5]:
print("Training Final Production Model")


final_train_data = lgb.Dataset(X_train, label=y_train)

final_model = lgb.train(
    config.params,
    final_train_data,
    num_boost_round=150
)



Training Final Production Model
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4474
[LightGBM] [Info] Number of data points in the train set: 140092, number of used features: 15
[LightGBM] [Info] Start training from score 2.639312


In [6]:
joblib.dump(final_model, MODELS_DIR / 'MarketDemandModel.joblib')

['C:\\Users\\UserB\\Documents\\ML\\Online Retail II UCI\\models\\MarketDemandModel.joblib']